# Recipe GenAI System — Data Cleaning

This notebook walks through the data ingestion and cleaning pipeline step by step.
Run cells sequentially after placing your dataset in `data/raw/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../'))

import pandas as pd
import matplotlib.pyplot as plt
from loguru import logger

from src.ingestion.load_data import load_dataset
from src.preprocessing.clean_text import clean_recipe_dataframe
from src.preprocessing.ingredient_parser import normalise_ingredient, normalise_ingredient_list
from src.utils.config import settings

## 1. Load Raw Data

In [ ]:
df_raw = load_dataset(source='auto')
print(f'Loaded {len(df_raw):,} recipes')
df_raw.head(3)

In [ ]:
# Dataset shape and dtypes
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())

## 2. Ingredient Distribution

In [ ]:
df_raw['n_ingredients'] = df_raw['ingredients'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df_raw['n_ingredients'].clip(upper=40).hist(bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Ingredient count distribution')
axes[0].set_xlabel('Number of ingredients')

df_raw['n_ingredients'].describe().to_frame().plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Ingredient count statistics')

plt.tight_layout()
plt.show()

print(df_raw['n_ingredients'].describe())

## 3. Clean Text

In [ ]:
df_clean = clean_recipe_dataframe(df_raw)
print(f'After cleaning: {len(df_clean):,} recipes')
df_clean[['clean_title','clean_instructions']].head(3)

## 4. Ingredient Normalisation

In [ ]:
# Side-by-side: raw vs normalised
sample = df_clean.iloc[0]['ingredients']
normed = normalise_ingredient_list(sample)

comparison = pd.DataFrame({'raw': sample, 'normalised': normed})
comparison

In [ ]:
# Most common normalised ingredients across dataset
from collections import Counter

all_ingredients = []
for ings in df_clean['ingredients']:
    if isinstance(ings, list):
        all_ingredients.extend(normalise_ingredient_list(ings))

top_50 = Counter(all_ingredients).most_common(50)
ing_df = pd.DataFrame(top_50, columns=['ingredient', 'count'])

ing_df.head(20).plot(
    kind='barh', x='ingredient', y='count',
    figsize=(10, 7), color='teal', legend=False
)
plt.title('Top 20 Most Common Ingredients')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Apply Ingredient Count Filters

In [ ]:
cfg = settings.preprocessing
df_filtered = df_clean[
    (df_clean['n_ingredients'] >= cfg.min_ingredients) &
    (df_clean['n_ingredients'] <= cfg.max_ingredients)
].reset_index(drop=True)

print(f'Before filter: {len(df_clean):,}')
print(f'After filter:  {len(df_filtered):,}')

## 6. Save Processed Data

In [ ]:
out_path = settings.paths.processed_data / 'recipes_clean.parquet'
df_filtered.to_parquet(out_path, index=False)
print(f'Saved {len(df_filtered):,} recipes to {out_path}')